In [2]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# Find integrated output
possible_paths = [
    OUTPUT_DIR / "proposal2_integrated_l3_time_score.csv",
    BASE_DIR / "proposal2_integrated_l3_time_score.csv",
]

integrated_path = None
for p in possible_paths:
    if p.exists():
        integrated_path = p
        break

if integrated_path is None:
    matches = list(BASE_DIR.rglob("proposal2_integrated_l3_time_score*"))
    if matches:
        integrated_path = matches[0]

if integrated_path is None:
    raise FileNotFoundError("Could not find proposal2_integrated_l3_time_score.csv/xls")

df = pd.read_csv(integrated_path)

# Ensure numeric columns
for col in [
    "l3_score_before_time",
    "l3_score_after_proposal2_time",
    "acc_score_final",
    "usa_score_final"
]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Score delta
df["proposal2_delta"] = (
    df["l3_score_after_proposal2_time"] - df["l3_score_before_time"]
)

# Changed flag
if "proposal2_changed_score" in df.columns:
    df["proposal2_changed_score"] = df["proposal2_changed_score"].astype(str).str.lower().isin(["true", "1", "yes"])
else:
    df["proposal2_changed_score"] = df["proposal2_delta"].fillna(0) != 0

# -------------------------
# 1. Task-level roll-up
# -------------------------
task_key = "model_c_task" if "model_c_task" in df.columns else "task_title"

task_rollup = (
    df.groupby(task_key, dropna=False)
    .agg(
        n_rows=("l3_score_before_time", "size"),
        n_changed=("proposal2_changed_score", "sum"),
        mean_l3_before_time=("l3_score_before_time", "mean"),
        mean_l3_after_proposal2_time=("l3_score_after_proposal2_time", "mean"),
        mean_delta=("proposal2_delta", "mean"),
        mean_acc_score_final=("acc_score_final", "mean"),
        mean_usa_score_final=("usa_score_final", "mean"),
    )
    .reset_index()
)

task_rollup.to_csv(OUTPUT_DIR / "proposal2_rollup_task.csv", index=False)

# -------------------------
# 2. Participant-level roll-up
# -------------------------
group_cols = []
if "email" in df.columns:
    group_cols.append("email")
if "participant" in df.columns:
    group_cols.append("participant")

if not group_cols:
    raise ValueError("No email or participant column found for participant roll-up")

participant_rollup = (
    df.groupby(group_cols, dropna=False)
    .agg(
        n_rows=("l3_score_before_time", "size"),
        n_changed=("proposal2_changed_score", "sum"),
        mean_l3_before_time=("l3_score_before_time", "mean"),
        mean_l3_after_proposal2_time=("l3_score_after_proposal2_time", "mean"),
        mean_delta=("proposal2_delta", "mean"),
        mean_acc_score_final=("acc_score_final", "mean"),
        mean_usa_score_final=("usa_score_final", "mean"),
    )
    .reset_index()
)

participant_rollup.to_csv(OUTPUT_DIR / "proposal2_rollup_participant.csv", index=False)

# -------------------------
# 3. Project-level roll-up
# -------------------------
project_rollup = pd.DataFrame([{
    "total_rows": len(df),
    "total_changed_rows": int(df["proposal2_changed_score"].sum()),
    "project_l3_before_time": df["l3_score_before_time"].mean(),
    "project_l3_after_proposal2_time": df["l3_score_after_proposal2_time"].mean(),
    "project_delta": df["proposal2_delta"].mean(),
    "project_acc_score_final": df["acc_score_final"].mean() if "acc_score_final" in df.columns else None,
    "project_usa_score_final": df["usa_score_final"].mean() if "usa_score_final" in df.columns else None,
}])

project_rollup.to_csv(OUTPUT_DIR / "proposal2_rollup_project.csv", index=False)

print("Generated:")
print(OUTPUT_DIR / "proposal2_rollup_task.csv")
print(OUTPUT_DIR / "proposal2_rollup_participant.csv")
print(OUTPUT_DIR / "proposal2_rollup_project.csv")

display(task_rollup.head())
display(participant_rollup.head())
display(project_rollup)

Generated:
C:\Users\85791\outputs\proposal2_rollup_task.csv
C:\Users\85791\outputs\proposal2_rollup_participant.csv
C:\Users\85791\outputs\proposal2_rollup_project.csv


,model_c_task,n_rows,n_changed,mean_l3_before_time,mean_l3_after_proposal2_time,mean_delta,mean_acc_score_final,mean_usa_score_final
0,AAMI Website,68,8,74.748882,69.448882,-5.300000,74.382647,74.748882
1,Equal Access to Critical Information – Emerge...,17,1,77.276471,75.394118,-1.882353,81.400000,77.276471
2,Finding Answers,34,2,38.233529,34.998235,-3.235294,41.779412,38.233529
3,PDFs and Understanding Terms,17,1,50.136471,49.548235,-0.588235,53.458824,50.136471


,email,participant,n_rows,n_changed,mean_l3_before_time,mean_l3_after_proposal2_time,mean_delta,mean_acc_score_final,mean_usa_score_final
0,carlpatrickrobinson@gmail.com,carlpatrickrobinson,6,0,80.000,80.000,0.00,80.000,80.000
1,carlpatrickrobinson@gmail.com,NaN,2,0,40.000,40.000,0.00,40.000,40.000
2,fjone7@gmail.com,fjone7,8,0,72.550,72.550,0.00,76.625,72.550
3,gameoverdan@gmail.com,gameoverdan,8,0,24.375,24.375,0.00,25.000,24.375
4,ghum@me.com,ghum,8,4,59.675,39.625,-20.05,47.250,59.675


,total_rows,total_changed_rows,project_l3_before_time,project_l3_after_proposal2_time,project_delta,project_acc_score_final,project_usa_score_final
0,136,12,62.859441,59.091794,-3.767647,64.493529,62.859441
